In [2]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns
import gc


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [ ]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
# app_name = "montage-pegasus-2mass-2deg-4node" #cosmoflow cm1
# app_name = "1000_genome_pegasus_node_16"
# app_name = "cm1"
# app_name = "deepspeed-dlio-step100"
# app_name = "deepspeed-dlio-scr-step100"
# app_name = "bert"
# app_name = "unet3d"
app_name = "montage_pegasus-dss-1deg_node-16"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint"

condition_fn = None #

if app_name == "montage_pegasus-dss-1deg_node-16":
    # filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/COMPACT/*.pfw.gz"
    filename = "/p/lustre3/iopp/v1.0.15.dev12/corona/montage/pegaus_dss_1deg/nodes-16/20251006193500/COMPACT/*.pfw.gz"
    
else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [07:51:41] Initialized Client with 192 workers and link http://134.9.71.27:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [07:51:49] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


2026-01-28 08:24:03,113 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [4]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))


In [5]:
def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    return d

load_cols = {'size': "int64[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}

In [6]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)

[INFO] [07:52:27] Created index for 14 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [07:52:27] Total size of all files are <dask.bag.core.Item object at 0x1554a707ef70> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [07:52:27] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [07:52:30] Loading 2013 batches out of 14 files and has 32884998 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [07:53:44] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [07:53:44] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [6]:
dir_w = "/p/lustre3/pandey2/logs/Results_Checkpoint/fhash/"+app_name+"/"
os.makedirs(dir_w, exist_ok=True)
analyzer.file_hash.reset_index().to_parquet(f'{dir_w}',engine='pyarrow',write_index=False)

In [7]:
analyzer.summary()

[INFO] [07:53:51] Total number of events in the workload are 32378942 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:623]


╭──────────────────────────────────────────────────── Summary ────────────────────────────────────────────────────╮
│  Allocation    Scheduler Allocation Details                                                                     │
│                ├── Nodes: 16                                                                                    │
│                ├── Processes: 921                                                                               │
│                ├── Thread allocations across nodes (includes dynamically created threads)                       │
│                │   ├── Compute: 0                                                                               │
│                │   └── I/O: 899                                                                                 │
│                └── Events Recorded: 32M                                                                         │
│  Dataset       Description of Dataset Used                                                                      │
│                └── Files: 14086                                                                                 │
│  I/O Behavior  Behavior of Application                                                                          │
│                ├── Split of Time in application                                                                 │
│                │   ├── Total Time: 2006.429 sec                                                                 │
│                │   └── Overall I/O: 24.049 sec                                                                  │
│                └── Metrics by function                                                                          │
│                    ├── Function       |count |                  size                   |                        │
│                    ├──                |      |min   |25    |mean  |median|75    |max   |                        │
│                    ├── opendir        |105K  |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── access         |9M    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── __xstat64      |5K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── open64         |2K    |4     |4     |4     |4     |4     |4     |                        │
│                    ├── lseek64        |2K    |NA    |NA    |NA    |NA    |NA    |NA    |                        │
│                    ├── read           |183K  |NA    |4     |811   |7     |27    |16KB  |                        │
│                    ├── close          |179K  |NA    |NA    |NA    |NA    |NA    |NA    |                        │
│                    ├── __xstat        |13K   |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── open           |120K  |4     |7     |7     |7     |7     |20    |                        │
│                    ├── __fxstat       |14K   |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── mmap           |15K   |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── unlink         |8K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── fork           |152   |143KB |521KB |734KB |835KB |835KB |835KB |                        │
│                    ├── write          |58    |246   |247   |248   |248   |248   |248   |                        │
│                    ├── remove         |90    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── ftruncate      |256   |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── fcntl          |1K    |NA    |nan   |nan   |NA    |nan   |NA    |                        │
│                    ├── openat         |92K   |-1    |-

In [11]:
if app_name in ["deepspeed-dlio-step100","deepspeed-dlio-scr-step100","resnet50","unet3d"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']]
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name == "montage-pegasus-2mass-2deg-4node":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name == "montage_pegasus-dss-1deg_node-16":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["1000_genome_pegasus_node_16"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]


elif app_name == "bert":
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]

elif app_name in ["cm1"]:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df1['fhash'] = df1['fhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df1['hhash'] = df1['hhash'].str.replace('.0', '', regex=False)  # Remove '.0'
    df3 = analyzer.host_hash.reset_index()[['hash', 'name']] 
    df3 = df3.rename(columns={'hash':'hhash'}) # update needed for this specific dataset
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname']] 
    analyze_df['id'] = analyze_df.index
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre3")
    data_calls = ['read', 'write', 'fread', 'fwrite', 'pread', 'pwrite']
    data_df = analyze_df[analyze_df["name"].isin(data_calls)]
    metadata_df = analyze_df[~analyze_df["name"].isin(data_calls)]




In [12]:
# Intermediate Steps Required 
# Step 1: Filter correct rows and categorize based on the size

bins = [2**i for i in range(0, 21)] + [float('inf')] 
bin_labels = [
    f"2^{i} - 2^{i+1} B" if i < 10 else
    f"2^{i-10} KiB - 2^{i-9} KiB" if i < 20 else
    "1 MiB+"
    for i in range(0, 21)
]

def categorize_sizes(df):
    df['size_category'] = pd.cut(df['size'], bins=bins, labels=bin_labels)
    return df

def categorize_sizes_metadata(df):
    df['size_category'] = "size"
    return df

data_df = data_df[data_df["size"] > 1]
data_df = data_df.map_partitions(categorize_sizes)
data_df['size_category'] = data_df['size_category'].astype('string[pyarrow]')

metadata_df = metadata_df.query("dur > 0")
metadata_df["size"] = 0 
metadata_df = metadata_df.map_partitions(categorize_sizes_metadata)
metadata_df['size_category'] = metadata_df['size_category'].astype('string[pyarrow]')


# # step 2: Filter only interesting mount points
# # if computationally expensive and only few 
# if app_name == "resnet500":
#     mount_point_list = ["/p/lustre3","/dev","/proc","/sys","/dev","/dev/shm","/usr/workspace"]
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# # if app_name == "deepspeed-dlio-scr-step100":
# #     mount_point_list = ['/l/ssd', '/p/lustre3','/usr/workspace', '/usr/WS2', '/sys', '/collab/usr/gapps', '/dev/shm','/dev','/proc', '/usr/tce','/usr/share','/var/tmp', '/g/g92']
# #     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
# #     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == 'montage-pegasus-2mass-2deg-4node':
#     mount_point_list = ['/p/lustre3', '/proc', '/usr/WS2', '/dev/shm', '/dev']
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == 'cm1':
#     mount_point_list =  ['/dev','/dev/shm', '/etc/libibverbs.d', '/p/lustre3','/sys','/tmp', '/usr/tce','/var/tmp', '/proc', '/usr/lib64', '/etc/psm3.conf', '/var']  
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]

# if app_name == '1000_genome_pegasus_node_16':
#     mount_point_list =  ['/p/lustre3', '/usr/WS2', '/usr/workspace', '/dev', '/proc', '/sys']
#     data_df = data_df[data_df["mount_point"].isin(mount_point_list)]
#     metadata_df = metadata_df[metadata_df["mount_point"].isin(mount_point_list)]
    


 

In [16]:
data_df['mount_point'] = data_df['mount_point'].replace('/p/lustre2', '/p/lustre3')
data_df.groupby("mount_point").count().compute()

,name,cat,size,ts,te,dur,trange,hostname,id,size_category
mount_point,,,,,,,,,,
/p/lustre3,1613342,1613342,1613342,1613342,1613342,1613342,1613342,1613342,1613342,1613342
/sys,112128,112128,112128,112128,112128,112128,112128,112128,112128,112128
/tmp,10528,10528,10528,10528,10528,10528,10528,10528,10528,10528
/,55504,55504,55504,55504,55504,55504,55504,55504,55504,55504
/dev,1,1,1,1,1,1,1,1,1,1


## Data Interference calculation

In [17]:
#DATA Interference Computation
IFCalculator = DFGrepInterferencePartitionBased(data_df, app_name=app_name, operation="data", cp_dir=cp_dir, existing=False)


In [18]:
IFCalculator.computeDegree()


In [19]:
IFCalculator.computeInterferenceData() 

[INFO] [07:38:04] Computing duration of minimum degree event for all degrees. [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph1.py:826]


[INFO] [07:38:26] Computing IF for all  [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/graph1.py:839]


In [ ]:
IFCalculator.inter.query()

In [12]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').groupby('mount_point').count().compute()

,name,cat,size,size_category,ts,te,dur,trange,hostname,deg_caller,deg_other,min_dur,interference
mount_point,,,,,,,,,,,,,
/dev,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506,9506
/dev/shm,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868,28868
/p/lustre3,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061,152061
/proc,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766,43766
/sys,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830,55830
/tmp,49,49,49,49,49,49,49,49,49,49,49,49,49
/usr/tce,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725,2725
/var,24,24,24,24,24,24,24,24,24,24,24,24,24
/var/tmp,310,310,310,310,310,310,310,310,310,310,310,310,310


In [20]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').groupby('mount_point').count().compute()

,name,cat,size,size_category,ts,te,dur,trange,hostname,deg_caller,deg_other,min_dur,interference
mount_point,,,,,,,,,,,,,
/,3621,3621,3621,3621,3621,3621,3621,3621,3621,3621,3621,3621,3621
/tmp,7947,7947,7947,7947,7947,7947,7947,7947,7947,7947,7947,7947,7947
/sys,83748,83748,83748,83748,83748,83748,83748,83748,83748,83748,83748,83748,83748
/p/lustre3,340574,340574,340574,340574,340574,340574,340574,340574,340574,340574,340574,340574,340574


In [22]:
IFCalculator.write_checkpoint(id='inter', cp_dir=cp_dir)

In [14]:
IFCalculator.inter.query('interference > 0 and deg_caller > 1').compute()

,name,cat,size,size_category,ts,te,dur,trange,mount_point,hostname,deg_caller,deg_other,min_dur,interference
69,read,POSIX,2,2^0 - 2^1 B,1276648,1276695,47,1,/,corona234,2,1,17,0.361702
70,read,POSIX,2,2^0 - 2^1 B,1276686,1276725,39,1,/,corona240,2,1,17,0.435897
94,read,POSIX,2,2^0 - 2^1 B,1314252,1314306,54,1,/,corona233,2,1,17,0.314815
95,read,POSIX,2,2^0 - 2^1 B,1314262,1314299,37,1,/,corona242,2,1,17,0.459459
104,read,POSIX,27,2^4 - 2^5 B,1333208,1333265,57,1,/,corona242,2,1,15,0.263158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7131,fread,STDIO,28800,2^4 KiB - 2^5 KiB,1919527910,1919527930,20,1919,/p/lustre3,corona229,2,1,6,0.3
7132,fread,STDIO,28800,2^4 KiB - 2^5 KiB,1919528694,1919528713,19,1919,/p/lustre3,corona229,2,1,6,0.315789
7133,fread,STDIO,28800,2^4 KiB - 2^5 KiB,1919529494,1919529514,20,1919,/p/lustre3,corona229,2,1,6,0.3
7134,fread,STDIO,28800,2^4 KiB - 2^5 KiB,1919530276,1919530295,19,1919,/p/lustre3,corona229,2,1,6,0.315789


# Metadata Interference Computation

In [16]:
IFCalculator_Metadata = DFGrepInterference(metadata_df, app_name=app_name, operation="metadata", cp_dir=cp_dir, existing=False)
# IFCalculator_Metadata.get_degree()
# IFCalculator_Metadata.compute_interference()
# IFCalculator_Metadata.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

In [17]:
IFCalculator_Metadata.get_degree()

In [ ]:
IFCalculator_Metadata.compute_interference()

In [ ]:
IFCalculator_Metadata.write_checkpoint(id="inter_metadata", cp_dir = cp_dir)

,name,pid,tid,hhash
hash,,,,
1.181405777718347e+19,corona188,1027673,1027673,11814057777183469021
6.968018510730724e+18,corona239,2055310,2055310,6968018510730723892
6.598453666498801e+18,corona234,2382606,2382606,6598453666498800848
1.0812942414789165e+19,corona236,269213,269213,10812942414789164657
1.1063386352187771e+18,corona238,2440296,2440296,1106338635218777131


# Burstiness Computation

In [23]:
delta = data_df.dur.max().compute()
BurstCalculator = DFGrepBurstiness(data_df, app_name=app_name, operation="data", delta = delta, cp_dir=cp_dir, existing=False)
BurstCalculator.compute_burstiness()
# BurstCalculator.write_checkpoint(id="ddf_bur",cp_dir=cp_dir)

In [24]:
BurstCalculator.write_checkpoint(id="ddf_bur",cp_dir=cp_dir)

In [25]:
BurstCalculator.ddf_bur.head()

,id,name,cat,size,ts,te,dur,trange,mount_point,hostname,group_key,b_id
0,413548,read,POSIX,8192,112437551,112437631,80,112,/tmp,corona232,"(112, '/tmp', 'corona232')",413548
1,163953,read,POSIX,8192,112438020,112438067,47,112,/tmp,corona232,"(112, '/tmp', 'corona232')",413548
2,197890,read,POSIX,8192,112438279,112438335,56,112,/tmp,corona232,"(112, '/tmp', 'corona232')",413548
3,450444,read,POSIX,8192,112438416,112438475,59,112,/tmp,corona232,"(112, '/tmp', 'corona232')",413548
4,441084,read,POSIX,8192,112438475,112438542,67,112,/tmp,corona232,"(112, '/tmp', 'corona232')",413548


<!-- delta = data_df.dur.max().compute() -->
